# m5 · Summary Report & Run Log
Aggregates all samples into a completion table, plots combined pLDDT distributions,
and writes `pipeline_run_log.txt`.

> **Standalone**: mount Drive, set parameters below, Run All.  
> **Via controller**: parameters are injected by papermill.

In [ ]:
# parameters
DRIVE_OUTPUT   = "mmpR5_pipeline/output"   # root inside MyDrive
REPORT_TITLE   = "mmpR5 Pipeline Run Report"
MAX_PER_FIGURE = 12                        # max pLDDT panels per figure

In [ ]:
# CPU only — no GPU needed
from google.colab import drive
drive.mount("/content/drive")
DRIVE_BASE  = Path("/content/drive/MyDrive/ColabNotebooks")
OUTPUT_ROOT = DRIVE_BASE / _p("DRIVE_OUTPUT", "mmpR5_pipeline/output")
REF_DIR     = DRIVE_BASE / _p("DRIVE_REF",    "mmpR5_pipeline/input")
MODULES_DIR = DRIVE_BASE / "mmpR5_pipeline" / "modules"
print(f"Drive mounted. Output root: {OUTPUT_ROOT}")

In [ ]:
# CPU only — no GPU needed
# ── Load pipeline config (Drive JSON fallback) ──────────────────────────────
import json, shutil, subprocess, warnings, time, datetime, concurrent.futures
from pathlib import Path
from Bio import SeqIO, Entrez
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import pandas as pd
import numpy as np
import requests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

_CFG_PATH = Path("/content/drive/MyDrive/ColabNotebooks/mmpR5_pipeline/pipeline_config.json")

def _load_config():
    if _CFG_PATH.exists():
        with open(_CFG_PATH) as _f:
            return json.load(_f)
    return {}

_cfg = _load_config()

def _p(key, default=None):
    """Resolve parameter: papermill-injected variable takes precedence over config JSON."""
    try:
        v = eval(key)                # injected by papermill
        return v if v is not None else _cfg.get(key, default)
    except Exception:
        return _cfg.get(key, default)

In [ ]:
import datetime, textwrap, math
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from pathlib import Path

DRIVE_OUTPUT   = _p("DRIVE_OUTPUT",   "mmpR5_pipeline/output")
REPORT_TITLE   = _p("REPORT_TITLE",   "mmpR5 Pipeline Run Report")
MAX_PER_FIGURE = int(_p("MAX_PER_FIGURE", 12))

OUTPUT_ROOT  = DRIVE_BASE / DRIVE_OUTPUT
SAMPLES_DIR  = OUTPUT_ROOT / "samples"
REPORTS_DIR  = OUTPUT_ROOT / "06_reports"; REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output root : {OUTPUT_ROOT}")
print(f"Samples dir : {SAMPLES_DIR}")

## Completion Table

In [ ]:
_STAGES = ["fastp", "alignment", "variants", "annotation", "colabfold",
           "mcsm", "maestro", "summary"]

_rows = []
for _sdir in sorted(SAMPLES_DIR.iterdir()) if SAMPLES_DIR.exists() else []:
    if not _sdir.is_dir():
        continue
    _label = _sdir.name
    _row = {"sample": _label}
    for _stage in _STAGES:
        # heuristic: check for key output files per stage
        _checks = {
            "fastp":     list(_sdir.glob("01_fastp/*.json")),
            "alignment": list(_sdir.glob("02_alignment/*.bam")),
            "variants":  list(_sdir.glob("03_variants/*.vcf*")),
            "annotation":list(_sdir.glob("04_annotation/*_variants_annotated.csv")),
            "colabfold": list(_sdir.glob("05_structure/**/*.pdb")),
            "mcsm":      list(_sdir.glob("04_annotation/*_mcsm*.csv")),
            "maestro":   list(_sdir.glob("04_annotation/*_maestro*.csv")),
            "summary":   list(_sdir.glob("*_full_summary.csv")),
        }
        _row[_stage] = "✓" if _checks.get(_stage) else "✗"
    _rows.append(_row)

if _rows:
    _comp_df = pd.DataFrame(_rows)
    print(f"Samples found: {len(_comp_df)}")
    display(_comp_df)
    _comp_df.to_csv(str(REPORTS_DIR / "completion_table.csv"), index=False)
    print(f"Saved → {REPORTS_DIR / 'completion_table.csv'}")
else:
    _comp_df = pd.DataFrame()
    print("[WARN] No sample directories found under", SAMPLES_DIR)

## Feature Table Summary

In [ ]:
_ft_path = OUTPUT_ROOT / "feature_table_all_samples_with_ml.csv"
if not _ft_path.exists():
    _ft_path = OUTPUT_ROOT / "feature_table_all_samples.csv"

if _ft_path.exists():
    _ft = pd.read_csv(str(_ft_path))
    print(f"Feature table: {len(_ft)} rows × {len(_ft.columns)} cols  ({_ft_path.name})")
    _phen = _ft["phenotype"].value_counts() if "phenotype" in _ft.columns else pd.Series(dtype=int)
    if not _phen.empty:
        print("Phenotype breakdown:", _phen.to_dict())
    _with_struct = _ft["has_structural_data"].sum() if "has_structural_data" in _ft.columns else 0
    print(f"Rows with structural data: {_with_struct}/{len(_ft)}")
    display(_ft.describe(include="all").loc[["count","unique","mean","std","min","max"]].fillna(""))
else:
    _ft = pd.DataFrame()
    print("[SKIP] No feature table found — run m3 first")

## Combined pLDDT Distributions
Max `MAX_PER_FIGURE` panels per figure.

In [ ]:
_plddt_files = []
for _sdir in sorted(SAMPLES_DIR.iterdir()) if SAMPLES_DIR.exists() else []:
    for _jf in sorted((_sdir / "05_structure").glob("**/*_scores_rank_001_*.json")) if (_sdir / "05_structure").exists() else []:
        _plddt_files.append((_sdir.name, _jf))

if not _plddt_files:
    print("[SKIP] No pLDDT score files found")
else:
    import json as _json
    _all_data = []
    for _label, _jf in _plddt_files:
        try:
            _d = _json.loads(_jf.read_text())
            _plddt = _d.get("plddt") or _d.get("confidences", {}).get("plddt")
            if _plddt:
                _all_data.append((_label, _plddt))
        except Exception as _e:
            print(f"  [WARN] {_jf.name}: {_e}")

    _n_total = len(_all_data)
    _n_figs  = math.ceil(_n_total / MAX_PER_FIGURE)
    print(f"Plotting {_n_total} pLDDT profiles across {_n_figs} figure(s)...")

    for _fi in range(_n_figs):
        _chunk = _all_data[_fi*MAX_PER_FIGURE : (_fi+1)*MAX_PER_FIGURE]
        _nc    = min(3, len(_chunk))
        _nr    = math.ceil(len(_chunk) / _nc)
        _fig, _axes = plt.subplots(_nr, _nc, figsize=(_nc*4, _nr*3), squeeze=False)
        for _ai, (_lbl, _pv) in enumerate(_chunk):
            _ax = _axes[_ai//_nc][_ai%_nc]
            _ax.plot(range(1, len(_pv)+1), _pv, color="#003f88", linewidth=0.8)
            _ax.axhline(70, color="orange", linestyle="--", linewidth=0.7, label="pLDDT=70")
            _ax.set_title(_lbl, fontsize=8)
            _ax.set_ylim(0, 100)
            _ax.set_xlabel("Residue", fontsize=7)
            _ax.set_ylabel("pLDDT", fontsize=7)
        # hide empty panels
        for _ai in range(len(_chunk), _nr*_nc):
            _axes[_ai//_nc][_ai%_nc].set_visible(False)
        plt.suptitle(f"pLDDT per residue — figure {_fi+1}/{_n_figs}", fontsize=10)
        plt.tight_layout()
        _png = REPORTS_DIR / f"plddt_combined_fig{_fi+1:02d}.png"
        plt.savefig(str(_png), dpi=150)
        plt.show(); plt.close()
        print(f"  Saved → {_png}")

## ML Results Summary

In [ ]:
_ml_dir = OUTPUT_ROOT / "07_ml_results"
_ml_csv = _ml_dir / "ml_cv_metrics.csv"
if _ml_csv.exists():
    _ml = pd.read_csv(str(_ml_csv))
    print("ML cross-validation results:")
    display(_ml)
else:
    print("[SKIP] No ML results found — run m4 first")

## Pipeline Run Log

In [ ]:
import platform, sys

_log_lines = [
    f"{'='*60}",
    f"  {REPORT_TITLE}",
    f"  Generated : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"{'='*60}",
    "",
    f"Python    : {sys.version.split()[0]}",
    f"Platform  : {platform.platform()}",
    f"Output    : {OUTPUT_ROOT}",
    "",
]

_log_lines.append("── Sample Completion ─────────────────────────")
if not _comp_df.empty:
    for _, _r in _comp_df.iterrows():
        _stages_ok  = [s for s in _STAGES if _r.get(s) == "✓"]
        _stages_bad = [s for s in _STAGES if _r.get(s) != "✓"]
        _log_lines.append(f"  {_r['sample']:30s}  done={len(_stages_ok)}/{len(_STAGES)}")
        if _stages_bad:
            _log_lines.append(f"    missing: {', '.join(_stages_bad)}")
else:
    _log_lines.append("  (no samples found)")

_log_lines += ["", "── Feature Table ──────────────────────────────"]
if not _ft.empty:
    _log_lines.append(f"  Total rows       : {len(_ft)}")
    _log_lines.append(f"  With struct data : {_ft.get('has_structural_data', pd.Series()).sum()}")
else:
    _log_lines.append("  (not found)")

_log_lines += ["", "── ML Results ─────────────────────────────────"]
if _ml_csv.exists():
    for _, _r in _ml.iterrows():
        _log_lines.append(f"  {_r.get('model','?'):20s}  G={_r.get('gscore_mean',float('nan')):.3f}  AUC={_r.get('roc_auc_mean',float('nan')):.3f}")
else:
    _log_lines.append("  (not found — run m4)")

_log_lines += ["", "="*60]

_log_text = "\n".join(_log_lines)
print(_log_text)
_log_path = REPORTS_DIR / "pipeline_run_log.txt"
_log_path.write_text(_log_text)
print(f"\nRun log saved → {_log_path}")